# PathoVision AI: training on a Kaggle GPU

This notebook trains both models on real data and runs whole-slide inference on a held-out CAMELYON16 slide.

**Before running:** in the right-hand panel set *Accelerator* to **GPU T4 x2** or **P100**, and turn *Internet* **on** (phone-verified account required).

| Step | What | Time (T4, approx.) |
|---|---|---|
| 1 | Setup | 2 min |
| 2 | Download PCam (~7.5 GB) | 5-10 min |
| 3 | Train ResNet-18 patch classifier | 20-40 min |
| 4 | Download 6 tumour + 2 normal CAMELYON16 slides | 10-20 min |
| 5 | Extract patches, train U-Net | 30-45 min |
| 6 | Whole-slide inference + evaluation | 5-10 min |

Record the numbers this notebook prints (`metrics.json`, `report.json`). Those are the figures for your README and resume.

In [ ]:
# 1. Setup: get the code (replace with your own GitHub repo URL after you push it)
!git clone https://github.com/nirdoshk0228/pathovision-ai.git /kaggle/working/pathovision-ai
%cd /kaggle/working/pathovision-ai
!pip install -q openslide-python openslide-bin imagecodecs
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working /tmp | tail -2

In [ ]:
import os
DATA = "/tmp/data"          # large scratch space; /kaggle/working is kept for outputs only
os.environ["DATA"] = DATA

## 2. PatchCamelyon (327,680 patches, 96x96 px @ 10x, slide-disjoint train/valid/test)

In [ ]:
!bash scripts/download_data.sh pcam

## 3. Patch classifier

ImageNet-pretrained ResNet-18, AdamW + one-cycle LR, AMP, HED stain + D4 augmentation.
The threshold is chosen on **validation**, and metrics are reported on the untouched **test** split with 8-view TTA.
Start with `--max-train 100000` for a fast first run, then drop it to use all 262k patches.

In [ ]:
!python -m pathovision.training.train_classifier --data-format pcam_h5 --data-dir $DATA/pcam \
    --arch resnet18 --epochs 6 --batch-size 256 --lr 1e-3 --workers 4 --tta 8 \
    --out-dir /kaggle/working/runs/cls_resnet18

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open("/kaggle/working/runs/cls_resnet18/metrics.json"))["results"]
for split, r in m.items():
    print(f"{split:9s} AUC {r['roc_auc']:.4f}  acc {r['accuracy']:.4f}  precision {r['precision']:.4f}  recall {r['recall']:.4f}  F1 {r['f1']:.4f}")
display(Image("/kaggle/working/runs/cls_resnet18/roc_test.png"))

## 4. CAMELYON16 whole-slide images

Slides come from the AWS Open Data bucket `s3://camelyon-dataset` (no account needed). Each slide is 0.5-3 GB.

In [ ]:
!N_TUMOR=6 N_NORMAL=2 bash scripts/download_data.sh camelyon

## 5. Patch extraction + U-Net segmentation

Splits are **by slide**: one tumour slide for validation, one tumour slide and one normal slide held out completely for step 6.
Segmentation patches are 256x256 px @ 0.972 um/px (10x), the same resolution as the classifier.

In [ ]:
import glob, os
imgs = sorted(glob.glob(f"{DATA}/camelyon16/images/*.tif"))
tumour = [os.path.basename(p)[:-4] for p in imgs if "tumor" in p]
normal = [os.path.basename(p)[:-4] for p in imgs if "normal" in p]
HOLDOUT = [tumour[-1], normal[-1]]
VAL = tumour[-2]
os.makedirs(f"{DATA}/holdout", exist_ok=True)
for s in HOLDOUT:
    os.replace(f"{DATA}/camelyon16/images/{s}.tif", f"{DATA}/holdout/{s}.tif")
print("train/val slides:", [s for s in tumour + normal if s not in HOLDOUT], "| val:", VAL, "| held out:", HOLDOUT)
os.environ["VAL"] = VAL

In [ ]:
!python scripts/extract_patches.py --slides "$DATA/camelyon16/images/*.tif" \
    --xml-dir $DATA/camelyon16/annotations --out $DATA/patches --val-slides $VAL --test-slides none \
    --no-cls --seg-size 256 --seg-mpp 0.972 --seg-per-slide 600

In [ ]:
!python -m pathovision.training.train_segmentation --data-dir $DATA/patches/seg \
    --arch resnet34_unet --epochs 20 --batch-size 16 --lr 3e-4 --workers 4 --tta 8 \
    --out-dir /kaggle/working/runs/seg_resnet34

## 6. Whole-slide inference on the held-out slides

In [ ]:
!mkdir -p /kaggle/working/models
!cp /kaggle/working/runs/cls_resnet18/best.pt /kaggle/working/models/classifier.pt
!cp /kaggle/working/runs/seg_resnet34/best.pt /kaggle/working/models/segmenter.pt
!python scripts/evaluate_slides.py --slides "$DATA/holdout/*.tif" --xml-dir $DATA/camelyon16/annotations \
    --cls-ckpt /kaggle/working/models/classifier.pt --seg-ckpt /kaggle/working/models/segmenter.pt \
    --out /kaggle/working/outputs/holdout --workers 4

In [ ]:
import glob
for d in sorted(glob.glob("/kaggle/working/outputs/holdout/*/")):
    r = json.load(open(d + "report.json"))
    q, b = r["quantification"], r["benchmark"]
    print(os.path.basename(d[:-1]), "|", r["slide"]["width"], "x", r["slide"]["height"],
          "| tiles:", b["tiles_classified"], "|", b["classification_tiles_per_sec"], "tiles/s | total", b["total_seconds"], "s",
          "| peak GPU", b["peak_gpu_mb"], "MB | lesions:", q["n_lesions"], "| category:", q["slide_category"],
          "| Dice:", r.get("ground_truth", {}).get("dice"))
    display(Image(d + "heatmap_overlay.png", width=700))
    display(Image(d + "prediction_overlay.png", width=700))

In [ ]:
# Throughput vs batch size / workers (for the README performance table)
!python scripts/benchmark.py --slide $(ls $DATA/holdout/tumor_*.tif | head -1) \
    --cls-ckpt /kaggle/working/models/classifier.pt --batch-sizes 64 256 512 --workers 2 4

Download `pathovision_results.zip` from the Output panel, then put `models/` in your repo (or a GitHub release) and the screenshots in `assets/`.

In [ ]:
!cd /kaggle/working && zip -qr pathovision_results.zip models runs outputs && ls -lh pathovision_results.zip